## Project 2: Linear regression
###### Predict the CO2 Emission from each car by it's information based on FuelConsumption dataset

#### Importing required packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import cross_val_score

#### Load Dataset

In [ ]:
url = "https://raw.githubusercontent.com/alihussainmeer/Data-Analysis-to-predict-the-CO2-Emission/refs/heads/master/FuelConsumption.csv"
df = pd.read_csv(url)
df.sample(7)

#### Data Observing
##### General Data Information

In [ ]:
display(df.shape)
print("******************************************************************************")
display(df.info())
print("******************************************************************************")
display(df.describe())
print("******************************************************************************")
display(df.isnull().sum())

##### Non-numerical Data Information

In [ ]:
display(df.nunique())
print("******************************************************************************")
display(df["MAKE"].value_counts())
print("******************************************************************************")
display(df["VEHICLECLASS"].value_counts())
print("******************************************************************************")
display(df["TRANSMISSION"].value_counts())
print("******************************************************************************")
display(df["FUELTYPE"].value_counts())

##### **Target Information**

In [ ]:
display(df["CO2EMISSIONS"].describe())
print("******************************************************************************")
display(df["CO2EMISSIONS"].value_counts().sort_values())
print("******************************************************************************")
display(df["CO2EMISSIONS"].nunique())

#### In Next cell, we use ```corr()``` that shows us the correlation coefficient.
#### A positive correlation coefficient indicates a positive (direct) relationship, while a negative one indicates a negative (inverse) relationship.

In [ ]:
df[["ENGINESIZE", "CO2EMISSIONS"]].corr()

---
### **Making First Model (With 1 feature)**

#### So, "ENGINESIZE" has strong relation with Emissions,  
#### can we use it as our only feature and get results? Let's see

In [ ]:
feature = ["ENGINESIZE"]
target = "CO2EMISSIONS"

# there is no need for Null checking because data is clean

X = df[feature]
Y = df[target]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)

In [ ]:
regr = LinearRegression()
regr.fit(X_train, Y_train)

#### Now we have tot test the model

In [ ]:
Y_pred = regr.predict(X_test)

mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print("MAE: ", mae)
print("MSE: ", mse)
print("R2: ", r2)

In [ ]:
print("Coefficient:", regr.coef_[0])
print("Intercept:", regr.intercept_)

#### Visualization

In [ ]:
plt.scatter(X_test, Y_test, alpha=0.5, color="navy")
plt.plot(X_test, X_test * regr.coef_[0] + regr.intercept_, color="gold")
plt.show()

*NOTE*: `"ENGINESIZE"` has a good predict for just one feature. Let's try two now but first we have to find the **best correlated** features with our target.

### Correlation Check

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols].corr()["CO2EMISSIONS"].sort_values(ascending=False)

##### We already have a vhiecle-based feature. Let's try a fuel-based one, I use FUELCONSUMPTION_COMB

### **Making Seccond Model (With 2 Features)**

In [ ]:
features = ["ENGINESIZE", "FUELCONSUMPTION_COMB"]
target = "CO2EMISSIONS"

# this model has numbers only and for that, the don't need encoding
# there is also no need for Null checking because data is clean

X = df[features]
Y = df[target]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    random_state = 42,
    train_size = 0.8
)

In [ ]:
regr = LinearRegression()
regr.fit(X_train, Y_train)

In [ ]:
Y_pred = regr.predict(X_test)

mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)

print("MAE: ", mae)
print("MSE: ", mse)
print("R2: ", r2)

In [ ]:
print("Coefficient: ", regr.coef_)
print("Intercept: ", regr.intercept_)

*NOTE*: This model is much better, but now, we need to check `multicollinearity`,why?  
Because our two features may have relations with each other.

### **Multicollinearity Check**

In [ ]:
df[["ENGINESIZE", "FUELCONSUMPTION_COMB"]].corr()

***RESULT:* We have a strong correlation between these two features, but we can't remove anything yet.**  
*this two features have good prediction score (0.86), but correlation shows independence level is **not** so high, for now we just keep the features.*

### 
---
**Now we use a test to findout `"ENGINESIZE"` is useful or not.**  
We have to build a model just by `"FUELCONSUMPTION_COMB"`

In [ ]:
X = df[["FUELCONSUMPTION_COMB"]]
Y = df["CO2EMISSIONS"]

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    random_state=42,
    test_size=0.2
)
regr_COMB = LinearRegression()
regr_COMB.fit(X_train, Y_train)

mae_COMB = mean_absolute_error(Y_test, Y_pred)
mse_COMB = mean_squared_error(Y_test, Y_pred)
r2_COMB = r2_score(Y_test, Y_pred)


print("MAE: ", mae_COMB)
print("MSE: ", mse_COMB)
print("R2: ", r2_COMB)
print("------------------------------------")
print("Coefficient: ", regr_COMB.coef_)
print("Intercept: ", regr_COMB.intercept_)

#### **TEST RESULTS**
**This test proves that `"ENGINESIZE"` is a useful feature but, with `"FUELCONSUMPTION_COMB"` in features, it has no extra informations to add in our analyze, decision is remove it.**

### **Making Third Model** (add Cylinders as new feature)

In [ ]:
features = ["FUELCONSUMPTION_COMB", "CYLINDERS"]
target = "CO2EMISSIONS"

X = df[features]
Y = df[target]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    random_state=42,
    test_size=0.2
)

In [ ]:
regr.fit(X_train, Y_train)
Ypred = regr.predict(X_test)

mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)


print("MAE: ", mae)
print("MSE: ", mse)
print("R2: ", r2)
print("------------------------------------")
print("Coefficient: ", regr.coef_)
print("Intercept: ", regr.intercept_)

***RESULTS:* Column `"CYLINDERS"` just like the `"ENGINESIZE"` has no more informations than `"FUELCONSUMPTION_COMB"` and its just a Multicollinearity.**

### Can`"FUELCONSUMPTION_COMB"` represents all consumptions?

In [ ]:
df[numeric_cols].corr()

#### The answer is yes, it has highest correlation scores in almost every 'consumption' parameters.

### Non-Numerical columns

In [ ]:
df.groupby("FUELTYPE")["CO2EMISSIONS"].agg(
    ["count", "mean", "median"]
).sort_values("mean")

In [ ]:
df.groupby("VEHICLECLASS")["CO2EMISSIONS"].agg(
    ["count", "mean", "median"]
).sort_values("mean")

In [ ]:
df.groupby("MAKE")["CO2EMISSIONS"].agg(
    ["count", "mean", "median"]
).sort_values("mean")

In [ ]:
df.groupby("TRANSMISSION")["CO2EMISSIONS"].agg(
    ["count", "mean", "median"]
).sort_values("mean")

### Making Models of Non-Numerical Columns

#### **FUELTYPE**

In [ ]:
X = df[["FUELCONSUMPTION_COMB", "FUELTYPE"]]
Y = df["CO2EMISSIONS"]

X = pd.get_dummies(X, columns=["FUELTYPE"], drop_first=True, dtype=int)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    random_state=42,
    train_size=0.8
)

In [ ]:
regr_FUELTYPE = LinearRegression()
regr_FUELTYPE.fit(X_train, Y_train)
Y_pred = regr_FUELTYPE.predict(X_test)

mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)


print("FUELTYPE_MAE: ", mae)
print("FUELTYPE_MSE: ", mse)
print("FUELTYPE_R2: ", r2)
print("------------------------------------")
print("FUELTYPE_Coefficient: ", regr_FUELTYPE.coef_)
print("FUELTYPE_Intercept: ", regr_FUELTYPE.intercept_)

#### **VEHICLECLASS**

In [ ]:
X = df[["FUELCONSUMPTION_COMB", "VEHICLECLASS"]]
Y = df["CO2EMISSIONS"]

X = pd.get_dummies(X, columns=["VEHICLECLASS"], drop_first=True, dtype=int)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    random_state=42,
    train_size=0.8
)

In [ ]:
regr_VEHICLECLASS = LinearRegression()
regr_VEHICLECLASS.fit(X_train, Y_train)
Y_pred = regr_VEHICLECLASS.predict(X_test)

mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)


print("VEHICLECLASS_MAE: ", mae)
print("VEHICLECLASS_MSE: ", mse)
print("VEHICLECLASS_R2: ", r2)
print("------------------------------------")
print("VEHICLECLASS_Coefficient: ", regr_VEHICLECLASS.coef_)
print("VEHICLECLASS_Intercept: ", regr_VEHICLECLASS.intercept_)

#### **MAKE**

In [ ]:
X = df[["FUELCONSUMPTION_COMB", "MAKE"]]
Y = df["CO2EMISSIONS"]

X = pd.get_dummies(X, columns=["MAKE"], drop_first=True, dtype=int)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    random_state=42,
    train_size=0.8
)

In [ ]:
regr_MAKE = LinearRegression()
regr_MAKE.fit(X_train, Y_train)
Y_pred = regr_MAKE.predict(X_test)

mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)


print("MAKE_MAE: ", mae)
print("MAKE_MSE: ", mse)
print("MAKE_R2: ", r2)
print("------------------------------------")
print("MAKE_Coefficient: ", regr_MAKE.coef_)
print("MAKE_Intercept: ", regr_MAKE.intercept_)

#### **TRANSMISSION**

In [ ]:
X = df[["FUELCONSUMPTION_COMB", "TRANSMISSION"]]
Y = df["CO2EMISSIONS"]

X = pd.get_dummies(X, columns=["TRANSMISSION"], drop_first=True, dtype=int)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    random_state=42,
    train_size=0.8
)

In [ ]:
regr_TRANSMISSION = LinearRegression()
regr_TRANSMISSION.fit(X_train, Y_train)
Y_pred = regr_TRANSMISSION.predict(X_test)

mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)


print("TRANSMISSION_MAE: ", mae)
print("TRANSMISSION_MSE: ", mse)
print("TRANSMISSION_R2: ", r2)
print("------------------------------------")
print("TRANSMISSION_Coefficient: ", regr_TRANSMISSION.coef_)
print("TRANSMISSION_Intercept: ", regr_TRANSMISSION.intercept_)

### **Analyze with 3 Features**

In [ ]:
df.groupby(["FUELTYPE"])["FUELCONSUMPTION_COMB"].agg(
    ["count", "mean", "median", "min", "max"]
).sort_values("mean", ascending=False)

In [ ]:
df.groupby("FUELTYPE").apply(
    lambda x: x["FUELCONSUMPTION_COMB"].corr(x["CO2EMISSIONS"])
)

#### *RESULT: Each `FUELTYPE` has a linear rilation with the `COMB`*
#### **SO, Let's findout about the interaction between *`COMB and FUELTYPE`***

In [ ]:
X = df[["FUELCONSUMPTION_COMB", "FUELTYPE"]]

X = pd.get_dummies(
    X,
    columns=["FUELTYPE"],
    drop_first=True,
    dtype=int
)

In [ ]:
X

Making interaction columns:

In [ ]:
X["COMB_E"] = X["FUELCONSUMPTION_COMB"] * X["FUELTYPE_E"]
X["COMB_X"] = X["FUELCONSUMPTION_COMB"] * X["FUELTYPE_X"]
X["COMB_Z"] = X["FUELCONSUMPTION_COMB"] * X["FUELTYPE_Z"]

In [ ]:
X

In [ ]:
Y = df["CO2EMISSIONS"]

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    regr
    X,
    Y,
    random_state=42,
    test_size=0.2
)

In [ ]:
regr_Interaction = LinearRegression()
regr_Interaction.fit(X_train, Y_train)
Y_pred = regr_Interaction.predict(X_test)

In [ ]:
mae = mean_absolute_error(Y_test, Y_pred)
mse = mean_squared_error(Y_test, Y_pred)
r2 = r2_score(Y_test, Y_pred)


print("Interaction_MAE: ", mae)
print("Interaction_MSE: ", mse)
print("Interaction_R2: ", r2)
print("------------------------------------")
print("Interaction_Coefficient: ", regr_Interaction.coef_)
print("Interaction_Intercept: ", regr_Interaction.intercept_)

In [ ]:
df.groupby("FUELTYPE").apply(
    lambda x: np.polyfit(
        x["FUELCONSUMPTION_COMB"],
        x["CO2EMISSIONS"],
        1
    )
)

### **Cross Validation**

In [ ]:
regr = LinearRegression()
scores = cross_val_score(
    regr,
    X,
    Y,
    cv=5,
    scoring='r2'
)

print("Scores: ",scores)
print("------------------------------------")
print("Mean: %.4f" % scores.mean())
print("------------------------------------")
print("STD: %.4f" % scores.std())

#### *RESULT: Approved.*

### **Residuals Analyze**

In [ ]:
model = LinearRegression()
model.fit(X, Y)

Y_pred = model.predict(X)
residuals = Y - Y_pred

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(Y_pred, residuals, alpha=0.6)

plt.axhline(y=0, linestyle="--", color="red")

plt.xlabel("Predicted CO2 Emissions")
plt.ylabel("Residuals")
plt.title("Residual Plot")

plt.show()

In [ ]:
df_res = df[["FUELTYPE"]].copy()

df_res["Residual"] = residuals
df_res["Predicted"] = Y_pred

print(df_res.groupby("FUELTYPE")["Residual"].agg(
    ["mean", "std", "min", "max"]
))

## **Final Model**

In [44]:
X = df[["FUELCONSUMPTION_COMB", "FUELTYPE"]]
y = df["CO2EMISSIONS"]
X = pd.get_dummies(
    X,
    columns=["FUELTYPE"],
    drop_first=True,
    dtype="int8"
)
final_model = LinearRegression()

final_model.fit(X, y)

print("Intercept:", final_model.intercept_)
print("------------------------------------")
print("Coefficients:", final_model.coef_)
print("------------------------------------")
print("R²:", final_model.score(X, y))

Intercept: 41.533250358724786
------------------------------------
Coefficients: [  22.0746367  -152.08471093  -31.83313857  -30.7323094 ]
------------------------------------
R²: 0.988755373484965
